# ICD-O Diagnosis Group Mapper: Development & Testing Notebook
This notebook serves as the development environment for a Python-based script designed to map ICD-O 4-digit diagnosis codes (e.g., 9380/3) or diagnosis terms (e.g., Glioma, malignant) to their corresponding diagnosis groups.

- Auto-Detection: The script automatically identifies whether the user provided codes or terms by checking for the ####/# pattern or matching against a known list of ICD-O terms.

- Intelligent Mapping:
For codes: extracts the first three digits of a diagnosis code to find the parent group.
For terms: performs exact string matching against standardized ICD-O preferred and synonyms terms.

- Data Validation: Validates the input format and reports "Recognition Errors" for any rows that do not match the expected format, allowing for partial mapping.

- Standardized Output: Produces a four-column TSV containing the original diagnosis code and term alongside the mapped group code/range and group name.

# TO-DO Items
- add logger instead of print statement
- user input should be a TSV
- convert this to a .py script that accepts user_input parameter? (upload to input folder)
- we need to wrap it in a function?

### Data Loading

In [78]:
# load packages
import pandas as pd
import re
from datetime import datetime

In [69]:
#load files
code_mapping_file = pd.read_csv("../data/code_mapping_file.csv") # doesn't have synonyms (if the user supplies codes)
term_mapping_file = pd.read_csv("../data/term_mapping_file.csv") # has synonyms (if the user supplies terms)
user_input = pd.read_csv("../data/example.csv")

### Data Validation

In [70]:
data = user_input.iloc[:,0].astype(str).str.strip()
invalid_rows = []

code_pattern = r'^\d{4}/\d$'
term_list = term_mapping_file['term']

# if the user supplies codes
if data.str.match(code_pattern).any():
    mapping_file = code_mapping_file
    key = 'code'
    if data.str.match(code_pattern).all():
        print("✅ Validation Success: All entries are codes.")
    else:
        invalid_rows = data[~data.str.match(code_pattern)].tolist()
        print(f"❌ Recognition Error: Some rows do not follow ####/# format.")
        
# if the user supplies terms
elif data.isin(term_list).any():
    mapping_file = term_mapping_file
    key = 'term'
    if data.isin(term_list).all():
        print("✅ Validation Success: All entries are terms.")
    else: 
        invalid_rows = data[~data.isin(term_list)].tolist()
        print(f"❌ Recognition Error: Some terms do not match ICD-O exactly.")
        
# if user-supplied data is not recognized
else:
    print(f"❌ Recognition Error: No entries were recognized.")
    print(f" Codes must follow the format ####/#. Terms must match ICD-O terms exactly (including commas/caps/spaces).")       

if invalid_rows:
    print(f"Some entries were not recognized: {invalid_rows}")
    print(f"Please submit only one type of data per column: codes(numeric) or terms(words)") 

✅ Validation Success: All entries are codes.


### Group Information Lookup

In [71]:
# look up the group information
output = user_submission.merge(
    mapping_file[['code', 'term', 'range', 'group']],
    on = key,
    how = 'left'
    )

In [72]:
# remove duplicate columns
cols_to_drop = [col for col in output.columns if col.endswith('_y')]
output = output.drop(columns=cols_to_drop)
output.columns = [col.replace('_x', '') for col in output.columns]

In [73]:
output.head(2)

,code,term,range,group
0,8000/0,"Neoplasm, benign",800,"Neoplasms, NOS"
1,8000/0,"Tumor, benign",800,"Neoplasms, NOS"


### Reporting

In [74]:
total = len(output)
matched = len(output[output['group'].notna()])
print(f"\nSummary:")
print(f"Total entries: {total}")
print(f"Successfully mapped: {matched}")
print(f"Unmatched: {total - matched}")


Summary:
Total codes: 50
Successfully mapped: 50
Unmatched: 0


### Saving

In [79]:
# Generate a timestamp string (e.g., '20260302_1545')
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

# create output file name
tsv_filename = f'../output/output_{timestamp}.tsv'
csv_filename = f'../output/output_{timestamp}.csv'

# export the files
output.to_csv(tsv_filename, sep='\t', index=False)
output.to_csv(csv_filename, sep=',', index=False)

print(f"✅ Files exported successfully: {tsv_filename}")

✅ Files exported successfully: ../output/output_20260302_1602.tsv
